In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

In [2]:
def examine(df, rows=3):
    print("Total of {} rows & {} columns".format(df.shape[0],df.shape[1]))
    display(pd.DataFrame(list(df.columns),columns=["Columns Headers"]).reset_index())
    print("Top {} rows of the DF".format(rows))
    display(df.head(rows))
    
# Define a custom formatting function
def format_sci_notation(x):
    return f'{x:.6f}'

In [3]:
# This is the dataset with Outliers, also used for PowerBI Dashboard
dataDF = pd.read_csv('SNRE_Dataset_2024_Outliers.csv')
display(dataDF.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9969 entries, 0 to 9968
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Contract Date                               9969 non-null   object 
 1   Project Name                                9969 non-null   object 
 2   Street Name                                 9969 non-null   object 
 3   Area (sqft)                                 9969 non-null   float64
 4   Floor Level                                 9969 non-null   object 
 5   Type of Sale                                9969 non-null   object 
 6   Region                                      9969 non-null   object 
 7   Remaining Tenure (Years)                    9969 non-null   int64  
 8   Total Population                            9969 non-null   float64
 9   SORA (%)                                    9969 non-null   float64
 10  Real GDP (in

None

OUTLIERS FOR CONTINUOUS VARIABLES
--------------------
Let us do some data prep before we do some exploration.

In [4]:
outliersDF = dataDF[['Area (sqft)', 'Remaining Tenure (Years)', 'Total Population', 
                            'SORA (%)', 'Real GDP (in $Sm)', 'Total Multiple-User Factory Space (m2)',
                            'US Dollar (Singapore Dollar Per US Dollar)', 'TPI % Change',
                            'Price', 'Unit Price ($ psf)','lagged psf','Rental Index % change']]

display(outliersDF)

,Area (sqft),Remaining Tenure (Years),Total Population,SORA (%),Real GDP (in $Sm),Total Multiple-User Factory Space (m2),US Dollar (Singapore Dollar Per US Dollar),TPI % Change,Price,Unit Price ($ psf),lagged psf,Rental Index % change
0,1733.0,52,5535002.0,0.39,423444.10,2258000.0,1.37,4.0,866500.0,500.00,548.18,2.4
1,1981.0,39,5637022.0,1.66,521934.96,992000.0,1.38,30.7,1219000.0,615.35,651.34,-7.0
2,2454.0,26,5612253.0,0.52,458446.91,1497000.0,1.38,-3.3,950000.0,387.12,372.75,-9.0
3,1109.0,61,5469724.0,0.08,411286.49,2103000.0,1.27,6.8,953497.0,859.78,613.27,-0.3
4,1851.0,36,5453566.0,0.27,503862.12,1449000.0,1.34,17.1,850000.0,459.21,516.28,-9.1
...,...,...,...,...,...,...,...,...,...,...,...,...
9964,2153.0,37,5703569.0,1.64,481335.21,1803000.0,1.36,-0.1,1000000.0,464.47,703.43,-9.0
9965,3089.0,34,5637022.0,0.68,521934.96,992000.0,1.38,30.7,1080000.0,349.63,802.85,-7.0
9966,1098.0,49,5453566.0,0.22,503862.12,1449000.0,1.34,17.1,488000.0,444.44,564.97,-9.9
9967,926.0,53,5535002.0,0.22,423444.10,1990000.0,1.37,4.0,600000.0,647.95,653.35,1.9


In [5]:
def outlierBound(df,col):
    colDF = df[col]
    display(colDF.describe())
    upperBound = colDF.describe()[1] + (3 * colDF.describe()[2])
    lowerBound = colDF.describe()[1] - (3 * colDF.describe()[2])
    print('For the Column: {}\nValues outside of {} and {} is an Outlier'.format(col, lowerBound, upperBound))
    print('No. of Outliers (Upper Bound) in {}: {}'.format(col ,colDF[colDF > upperBound].count()))
    return lowerBound, upperBound

In [6]:
arealowerBound, areaupperBound = outlierBound(outliersDF,'Area (sqft)')

count     9969.000000
mean      2380.927776
std       2366.271583
min        398.000000
25%       1464.000000
50%       1755.000000
75%       2605.000000
max      82904.000000
Name: Area (sqft), dtype: float64

For the Column: Area (sqft)
Values outside of -4717.88697349252 and 9479.742525704376 is an Outlier
No. of Outliers (Upper Bound) in Area (sqft): 107


In [7]:
tenurelowerBound, tenureupperBound = outlierBound(outliersDF,'Remaining Tenure (Years)')

count    9969.000000
mean       37.391915
std        12.660803
min         3.000000
25%        27.000000
50%        38.000000
75%        49.000000
max        70.000000
Name: Remaining Tenure (Years), dtype: float64

For the Column: Remaining Tenure (Years)
Values outside of -0.5904932571283084 and 75.3743231297334 is an Outlier
No. of Outliers (Upper Bound) in Remaining Tenure (Years): 0


In [8]:
poplowerBound, popupperBound = outlierBound(outliersDF,'Total Population')

count    9.969000e+03
mean     5.569018e+06
std      9.427799e+04
min      5.399162e+06
25%      5.469724e+06
50%      5.607283e+06
75%      5.637022e+06
max      5.703569e+06
Name: Total Population, dtype: float64

For the Column: Total Population
Values outside of 5286183.577751487 and 5851851.537706434 is an Outlier
No. of Outliers (Upper Bound) in Total Population: 0


In [9]:
soralowerBound, soraupperBound = outlierBound(outliersDF,'SORA (%)')

count    9969.000000
mean        0.746896
std         0.891024
min         0.030000
25%         0.160000
50%         0.330000
75%         1.060000
max         4.390000
Name: SORA (%), dtype: float64

For the Column: SORA (%)
Values outside of -1.9261767103370406 and 3.4199694678854407 is an Outlier
No. of Outliers (Upper Bound) in SORA (%): 262


In [10]:
gdplowerBound, gdpupperBound = outlierBound(outliersDF,'Real GDP (in $Sm)')

count      9969.000000
mean     461882.058247
std       40634.911437
min      395550.150000
25%      423444.100000
50%      462647.930000
75%      503862.120000
max      521934.960000
Name: Real GDP (in $Sm), dtype: float64

For the Column: Real GDP (in $Sm)
Values outside of 339977.32393508847 and 583786.7925580402 is an Outlier
No. of Outliers (Upper Bound) in Real GDP (in $Sm): 0


In [11]:
spacelowerBound, spaceupperBound = outlierBound(outliersDF,'Total Multiple-User Factory Space (m2)')

count    9.969000e+03
mean     1.662311e+06
std      4.159921e+05
min      7.900000e+05
25%      1.449000e+06
50%      1.651000e+06
75%      1.990000e+06
max      2.360000e+06
Name: Total Multiple-User Factory Space (m2), dtype: float64

For the Column: Total Multiple-User Factory Space (m2)
Values outside of 414334.3764955045 and 2910286.7489935113 is an Outlier
No. of Outliers (Upper Bound) in Total Multiple-User Factory Space (m2): 0


In [12]:
usdlowerBound, usdupperBound = outlierBound(outliersDF,'US Dollar (Singapore Dollar Per US Dollar)')

count    9969.000000
mean        1.345440
std         0.043643
min         1.250000
25%         1.340000
50%         1.360000
75%         1.380000
max         1.380000
Name: US Dollar (Singapore Dollar Per US Dollar), dtype: float64

For the Column: US Dollar (Singapore Dollar Per US Dollar)
Values outside of 1.2145110964056731 and 1.4763686307485044 is an Outlier
No. of Outliers (Upper Bound) in US Dollar (Singapore Dollar Per US Dollar): 0


In [13]:
tpilowerBound, tpiupperBound = outlierBound(outliersDF,'TPI % Change')

count    9969.000000
mean        8.455653
std        11.093910
min        -3.300000
25%        -0.100000
50%         4.000000
75%        17.100000
max        30.700000
Name: TPI % Change, dtype: float64

For the Column: TPI % Change
Values outside of -24.826076771156963 and 41.73738181679845 is an Outlier
No. of Outliers (Upper Bound) in TPI % Change: 0


In [14]:
pricelowerBound, priceupperBound = outlierBound(outliersDF,'Price')

count    9.969000e+03
mean     8.129001e+05
std      6.087932e+05
min      6.500000e+04
25%      5.200000e+05
50%      6.800000e+05
75%      9.180000e+05
max      1.600000e+07
Name: Price, dtype: float64

For the Column: Price
Values outside of -1013479.4587554398 and 2639279.6784364507 is an Outlier
No. of Outliers (Upper Bound) in Price: 149


In [15]:
psflowerBound, psfupperBound = outlierBound(outliersDF,'Unit Price ($ psf)')

count    9969.000000
mean      385.040204
std       153.274868
min        38.050000
25%       271.670000
50%       370.850000
75%       463.950000
max      1800.000000
Name: Unit Price ($ psf), dtype: float64

For the Column: Unit Price ($ psf)
Values outside of -74.78439897348937 and 844.8648062360031 is an Outlier
No. of Outliers (Upper Bound) in Unit Price ($ psf): 199


In [16]:
laglowerBound, lagupperBound = outlierBound(outliersDF,'lagged psf')

count    9969.000000
mean      421.244153
std       162.321727
min        49.320000
25%       302.480000
50%       410.400000
75%       502.400000
max      1800.000000
Name: lagged psf, dtype: float64

For the Column: lagged psf
Values outside of -65.72102668511536 and 908.2093324329336 is an Outlier
No. of Outliers (Upper Bound) in lagged psf: 182


In [17]:
rentIndexlowerBound, rentIndexupperBound = outlierBound(outliersDF,'Rental Index % change')

count    9969.000000
mean       -7.806600
std         8.085403
min       -24.700000
25%       -11.100000
50%        -6.300000
75%        -1.600000
max         7.600000
Name: Rental Index % change, dtype: float64

For the Column: Rental Index % change
Values outside of -32.062809995563356 and 16.449609072702483 is an Outlier
No. of Outliers (Upper Bound) in Rental Index % change: 0


Removing Outliers from the original Dataset - dataDF
-------------------------------
Another way of doing it is Scipy.stat.zscore 
https://stackoverflow.com/questions/23199796/detect-and-exclude-outliers-in-a-pandas-dataframe

In [18]:
# Remember dataDF is the original dataset -> remove outliers from dataDF

#dataDF1 = remove Area(sqft) outliers first,Keeping values < 9474.75
#display(areaupperbound)
dataDF1 = dataDF[dataDF['Area (sqft)'] <= areaupperBound].reset_index(drop=True)
#display(dataDF1.shape)

#dataDF2 = remove Remaining Tenure outliers first,Keeping values < 74.46
#display(tenureupperbound)
dataDF2 = dataDF1[dataDF1['Remaining Tenure (Years)'] <= tenureupperBound].reset_index(drop=True)
#display(dataDF2.shape)

#dataDF3 = remove population outliers first,Keeping values < 5851826.057037627
#display(popupperbound)
dataDF3 = dataDF2[dataDF2['Total Population'] <= popupperBound].reset_index(drop=True)
#display(dataDF3.shape)

#dataDF4 = remove SORA first,Keeping values < 3.4255335798381807
#display(soraupperbound)
dataDF4 = dataDF3[dataDF3['SORA (%)'] <= soraupperBound].reset_index(drop=True)
#display(dataDF4.shape)

#dataDF5 = remove REAL GDP first,Keeping values < 1,466,560,549.1033294
#display(gdpupperbound)
dataDF5 = dataDF4[dataDF4['Real GDP (in $Sm)'] <= gdpupperBound].reset_index(drop=True)
#display(dataDF5.shape)

#dataDF6 = remove Supply of Land outliers first,Keeping values < 8057324.223879513
#display(supplyupperbound)
dataDF6 = dataDF5[dataDF5['Total Multiple-User Factory Space (m2)'] <= spaceupperBound].reset_index(drop=True)
#display(dataDF6.shape)

#dataDF7 = remove US dollar Outliers
dataDF7 = dataDF6[dataDF6['US Dollar (Singapore Dollar Per US Dollar)'] <= usdupperBound].reset_index(drop=True)


#dataDF8 = remove Material Cost outliers first,Keeping values < 484161681.3620674
#display(costupperbound)
dataDF8 = dataDF7[dataDF7['TPI % Change'] <= tpiupperBound].reset_index(drop=True)
#display(dataDF8.shape)

#dataDF9 = remove Price outliers first,Keeping values < 74.46
#display(priceupperbound)
dataDF9 = dataDF8[dataDF8['Price'] <= priceupperBound].reset_index(drop=True)

#dataDF10 = remove psf outliers first,Keeping values < 74.46
#display(priceupperbound)
dataDF10 = dataDF9[dataDF9['Unit Price ($ psf)'] <= psfupperBound].reset_index(drop=True)

#dataDF11 = remove lagged psf outliers first,Keeping values < 908.21.46
#display(priceupperbound)
dataDF11 = dataDF10[dataDF10['lagged psf'] <= lagupperBound].reset_index(drop=True)

#dataDF11 = remove Rental Index % Change outliers first,Keeping values < 16.45
#display(priceupperbound)
dataDF12 = dataDF11[dataDF11['Rental Index % change'] <= rentIndexupperBound].reset_index(drop=True)

In [19]:
# Remember the following - 
# dataDF - Original Dataset from .csv file with Outliers
# dataDF12 - Dataset with Outliers removed (3 s.d method)
display(dataDF12.info())
display("Before Outlier Removal - {}".format(dataDF.shape))
display("After Outlier Removal - {}".format(dataDF12.shape))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9297 entries, 0 to 9296
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Contract Date                               9297 non-null   object 
 1   Project Name                                9297 non-null   object 
 2   Street Name                                 9297 non-null   object 
 3   Area (sqft)                                 9297 non-null   float64
 4   Floor Level                                 9297 non-null   object 
 5   Type of Sale                                9297 non-null   object 
 6   Region                                      9297 non-null   object 
 7   Remaining Tenure (Years)                    9297 non-null   int64  
 8   Total Population                            9297 non-null   float64
 9   SORA (%)                                    9297 non-null   float64
 10  Real GDP (in

None

'Before Outlier Removal - (9969, 29)'

'After Outlier Removal - (9297, 29)'

In [20]:
# Using pandas.get_dummies to do dummy encoding
# https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html

# Reference Column is New Sale
sale_dummy = pd.get_dummies(dataDF12['Type of Sale'], drop_first=False, dtype = 'int64')

# Reference Column is First Floor
floor_dummy = pd.get_dummies(dataDF12['Floor Level'], drop_first=False, dtype = 'int64')

# Reference Column is Central Region
region_dummy = pd.get_dummies(dataDF12['Region'], drop_first=False, dtype = 'int64')

# Next i need to merge my dummy variables and remove the columns
sale_dummy = sale_dummy.join(floor_dummy).join(region_dummy)
x_dummy = dataDF12.join(sale_dummy)
x_dummy = x_dummy[['Floor Level', 'First Floor','Non-First Floor','Type of Sale','New Sale','Resale','Subsale',
                   'Region', 'Central Region','East Region', 'North Region','North-East Region', 'West Region',
                   'Remaining Tenure (Years)','Total Population','SORA (%)','Real GDP (in $Sm)',
                   'Total Multiple-User Factory Space (m2)','US Dollar (Singapore Dollar Per US Dollar)',
                   'TPI % Change','lagged psf','Rental Index % change','minDistToMRT','Unit Price ($ psf)']]

# notice the increment of 10 variables, these are the dummy variables
display(x_dummy.columns)
display(x_dummy.shape)


Index(['Floor Level', 'First Floor', 'Non-First Floor', 'Type of Sale',
       'New Sale', 'Resale', 'Subsale', 'Region', 'Central Region',
       'East Region', 'North Region', 'North-East Region', 'West Region',
       'Remaining Tenure (Years)', 'Total Population', 'SORA (%)',
       'Real GDP (in $Sm)', 'Total Multiple-User Factory Space (m2)',
       'US Dollar (Singapore Dollar Per US Dollar)', 'TPI % Change',
       'lagged psf', 'Rental Index % change', 'minDistToMRT',
       'Unit Price ($ psf)'],
      dtype='object')

(9297, 24)

In [21]:
display(x_dummy.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9297 entries, 0 to 9296
Data columns (total 24 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Floor Level                                 9297 non-null   object 
 1   First Floor                                 9297 non-null   int64  
 2   Non-First Floor                             9297 non-null   int64  
 3   Type of Sale                                9297 non-null   object 
 4   New Sale                                    9297 non-null   int64  
 5   Resale                                      9297 non-null   int64  
 6   Subsale                                     9297 non-null   int64  
 7   Region                                      9297 non-null   object 
 8   Central Region                              9297 non-null   int64  
 9   East Region                                 9297 non-null   int64  
 10  North Region

None

In [22]:
# # Prepare Dataset with dummy and no Outliers for SPSS
# x_dummy.to_csv("SNRE_Data_w_Dummy_for_SPSS.csv", index=False)

# #Testing Purpose
x_dummy.to_csv("SNRE_Data_w_Dummy_for_SPSS_6April.csv", index=False)